<h1> Formal law: Similarity </h1>

<ol type="1">
  <li>Convert melody to interval sequence</li>
  <li>Align sequences (global and local)</li>
  <li>Count matching intervals</li>
  <li>Percent Melodic Idendity (PMI)</li>
</ol>


<h2> Interval represenation </h2>

Instead of raw pitches (e.g., 60, 62, 64), we convert to intervals because copyright cases often care about relative melodic contour, not key. It captures the melodic structure and relative pitch movement.

<ol type="1">
  <li>Load MIDI</li>
  <li>Extract the melody (highest pitch per timestep OR single track)</li>
  <li>Convert it into a symbolic string sequence</li>
  <li>Return something ready for alignment (e.g., Levenshtein / DTW)</li>
</ol>

In [21]:
import pretty_midi

def extract_melody_intervals(midi_path):
    """
    Extracts a melody sequence from a MIDI file
    and returns a string of pitch intervals.
    """

    midi = pretty_midi.PrettyMIDI(midi_path)

    # Choose first instrument (simplest assumption). Manually ensure each file contains only the melody 
    # with SignalMIDI: https://signalmidi.app/?lang=en
    instrument = midi.instruments[0]

    # Get notes sorted by start time
    notes = sorted(instrument.notes, key=lambda n: n.start)

    if len(notes) < 2:
        return ""

    # Extract pitch sequence
    pitches = [note.pitch for note in notes]

    # Convert to interval sequence
    intervals = []
    for i in range(1, len(pitches)):
        interval = pitches[i] - pitches[i - 1]
        intervals.append(interval)

    return intervals



<h2> Similarity models </h2>

We implement two similarity models: a conservative baseline using global alignment and strict identity, and a more perceptually flexible model using local alignment and tolerant identity. The baseline provides a conservative similarity estimate, while the improved model captures fragment-level and perceptual similarity that may be legally relevant.

<h2> Baseline model </h2>
Full-excerpt comparison for strict identity measurement. <br> <br>
- Interval representation <br>
- Global alignment (Needleman–Wunsch) <br>
- Strict PMI <br>
- Full-phrase comparison <br>

In [8]:
def global_align(seq1, seq2):
    """
    Perform global alignment using simple scoring:
    match = 1
    mismatch = 0
    gap = 0
    
    Returns aligned sequences.
    """
    
    n = len(seq1)
    m = len(seq2)
    
    # Create scoring matrix initialized to zeros
    dp = [[0 for _ in range(m + 1)] for _ in range(n + 1)]
    
    # Fill dynamic programming matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            
            # match score: 1 if identical, else 0
            match = dp[i - 1][j - 1] + (1 if seq1[i - 1] == seq2[j - 1] else 0)
            
            # no gap penalty
            delete = dp[i - 1][j]
            insert = dp[i][j - 1]
            
            dp[i][j] = max(match, delete, insert)
    
    # Backtracking
    aligned1 = []
    aligned2 = []
    
    i, j = n, m
    
    while i > 0 and j > 0:
        if dp[i][j] == dp[i - 1][j - 1] + (1 if seq1[i - 1] == seq2[j - 1] else 0):
            aligned1.append(seq1[i - 1])
            aligned2.append(seq2[j - 1])
            i -= 1
            j -= 1
        elif dp[i][j] == dp[i - 1][j]:
            aligned1.append(seq1[i - 1])
            aligned2.append(None)
            i -= 1
        else:
            aligned1.append(None)
            aligned2.append(seq2[j - 1])
            j -= 1
    
    # reverse because we built alignment backwards
    aligned1.reverse()
    aligned2.reverse()
    
    return aligned1, aligned2

def strict_pmi(aligned1, aligned2):
    """
    Compute strict Percent Melodic Identity.
    Only exact matches count.
    """
    
    matches = 0
    length = (len(aligned1) + len(aligned2)) / 2
    
    for a, b in zip(aligned1, aligned2):
        if a is not None and b is not None and a == b:
            matches += 1
    
    PMI = (matches / length) * 100

    return PMI if length > 0 else 0

<h2> Improved model </h2>
Copyright law often focuses on copied fragments and substantial similarity may exist in part of a melody. So, a local alignment makes sense.<br> <br>
- Same representation <br>
- Local alignment (Smith–Waterman) <br>
- Tolerant PMI <br>
- Fragment-focused comparison <br>

In [23]:
def local_align(seq1, seq2):
    """
    Local alignment with tolerant scoring.
    
    Scoring:
    exact match = +2
    near match (±1) = +1
    mismatch = -1
    gap = -1
    """
    
    n = len(seq1)
    m = len(seq2)
    
    dp = [[0 for _ in range(m + 1)] for _ in range(n + 1)]
    
    max_score = 0
    max_pos = (0, 0)
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            
            diff = abs(seq1[i - 1] - seq2[j - 1])
            
            if diff == 0:
                score = 2
            elif diff == 1:
                score = 1
            else:
                score = -1
            
            match = dp[i - 1][j - 1] + score
            delete = dp[i - 1][j] - 1
            insert = dp[i][j - 1] - 1
            
            dp[i][j] = max(0, match, delete, insert)
            
            if dp[i][j] > max_score:
                max_score = dp[i][j]
                max_pos = (i, j)
    
    # Backtrack from max position
    aligned1 = []
    aligned2 = []
    
    i, j = max_pos
    
    while i > 0 and j > 0 and dp[i][j] > 0:
        diff = abs(seq1[i - 1] - seq2[j - 1])
        
        if diff == 0:
            score = 2
        elif diff == 1:
            score = 1
        else:
            score = -1
        
        if dp[i][j] == dp[i - 1][j - 1] + score:
            aligned1.append(seq1[i - 1])
            aligned2.append(seq2[j - 1])
            i -= 1
            j -= 1
        elif dp[i][j] == dp[i - 1][j] - 1:
            aligned1.append(seq1[i - 1])
            aligned2.append(None)
            i -= 1
        else:
            aligned1.append(None)
            aligned2.append(seq2[j - 1])
            j -= 1
    
    aligned1.reverse()
    aligned2.reverse()
    
    return aligned1, aligned2

def tolerant_pmi(aligned1, aligned2):
    """
    Tolerant PMI: counts matches where abs(a - b) <= 1
    """
    
    matches = 0
    length = len(aligned1)
    
    for a, b in zip(aligned1, aligned2):
        if a is not None and b is not None:
            if abs(a - b) <= 1:
                matches += 1
                
    PMI = (matches / length) * 100
    
    return PMI if length > 0 else 0

<h2> Implementation </h2>

In [28]:
#Interval representation
case120_A = extract_melody_intervals(r"C:\Users\gresi\Downloads\Ryad2 Formal_law main data-songs\120\The_Last_Time_Melody.mid")
case120_B = extract_melody_intervals(r"C:\Users\gresi\Downloads\Ryad2 Formal_law main data-songs\120\Bitter_Sweet_Symphony_Melody.mid")

#Baseline model
A1_global, A2_global = global_align(case120_A, case120_B)
Baseline_PMI = strict_pmi(A1_global, A2_global)

print(f"Baseline global PMI: {Baseline_PMI:.2f}%")

#Improved model
A1_local, A2_local = local_align(case120_A, case120_B)
Improved_PMI = tolerant_pmi(A1_local, A2_local)

print(f"Improved local PMI: {Improved_PMI:.2f}%")

Baseline global PMI: 20.43%
Improved local PMI: 51.72%
